# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

### Exploratory Data Analysis & Heavy Tails:
We look at the percentiles of our primary metrics across the 30,000 rows in our starter dataset:

- **`impressions_90d`**: Extremely heavy-tailed. Median is 731, but the 99th percentile is 73,505, and the max is 517,715. A few giant pages dominate the search impressions.
- **`clicks_90d`**: Even heavier tail. Median is just 1 click, the 99th percentile is 253, and the max is 4,178.
- **`avg_position`**: Median position is 10.8 (right at the bottom of Page 1 / top of Page 2).
- **`days_since_last_update`**: Concentrated heavily at 20 days and 104 days (representing systematic bulk updates), with a maximum of 373 days.
- **`word_count`**: Median is 2,877 words. Note that 7,699 rows (25.7%) are missing word counts. This is a key data limitation.

In [3]:
# Code cell 2: Print out the distribution percentiles of key fields
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

for col in ["impressions_90d", "clicks_90d", "avg_position", "days_since_last_update", "word_count"]:
    print(f"\n--- {col} distribution percentiles ---")
    desc = df[col].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99])
    print(desc.to_string())



--- impressions_90d distribution percentiles ---
count     30000.000000
mean       5200.366300
std       16838.019547
min           1.000000
25%          81.000000
50%         731.000000
75%        3615.250000
90%       12136.400000
99%       73505.830000
max      517715.000000

--- clicks_90d distribution percentiles ---
count    30000.000000
mean        16.097333
std         75.076958
min          0.000000
25%          0.000000
50%          1.000000
75%          7.000000
90%         32.000000
99%        253.010000
max       4178.000000

--- avg_position distribution percentiles ---
count    30000.00000
mean        16.34238
std         15.21679
min          0.00000
25%          6.20000
50%         10.80000
75%         22.30000
90%         36.80000
99%         69.90100
max        245.00000

--- days_since_last_update distribution percentiles ---
count    30000.000000
mean        46.098300
std         42.078709
min          1.000000
25%         20.000000
50%         20.000000
75%      

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

### Signal Mini-Tests & Verdicts:

1. **Signal 1: Volume (`impressions_90d`) vs. Decline**
   - *Claim:* High-volume pages are more stable and less likely to decline.
   - *Verdict:* **MIXED**
   - *Explanation:* The decline rate rises from 38.9% for low-volume (<100) pages to 62.0% for high-volume (1k-10k) pages, but then dips to 52.4% for very high-volume (10k+) pages. Low-volume pages are already at the bottom and have less room to decline, while very high-volume pages represent core stable content.

2. **Signal 2: Word Count (`word_count`) vs. Decline**
   - *Claim:* Thin content (<500 words) drops in ranking and has higher decline rates.
   - *Verdict:* **OPPOSITE / FALSE**
   - *Explanation:* Medium articles (500-1200 words) have a lower decline rate of 24.9% compared to long articles (1200+ words) which decline at 55.5%. Note that the "Thin (<500)" bucket only contains 3 rows in the dataset, which is below our sample size floor of 50. In this dataset, longer articles are more volatile and more likely to decay.

3. **Signal 3: Position (`avg_position`) vs. Decline**
   - *Claim:* Pages on Page 1 (1-10) are more stable than Page 2 (10-20) or Page 3+ (20+).
   - *Verdict:* **MIXED**
   - *Explanation:* Page 2 pages are the most volatile, with a 61.0% decline rate. Page 1 pages have a 56.3% decline rate, indicating they are in highly competitive positions where ranking decays are common.

In [5]:
# Code cell 4: Output the signal bucket tables
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

print("--- SIGNAL 1: Volume Buckets vs. Decline ---")
def get_vol_bucket(imp):
    if imp < 100: return "1. <100 (Low)"
    elif imp < 1000: return "2. 100-1000 (Medium)"
    elif imp < 10000: return "3. 1000-10k (High)"
    else: return "4. 10k+ (Very High)"

df["vol_bucket"] = df["impressions_90d"].apply(get_vol_bucket)
t1 = df.groupby("vol_bucket")["is_declining_label"].agg(["count", "mean"])
print(t1.to_string())

print("\n--- SIGNAL 2: Word Count Buckets vs. Decline ---")
def get_wc_bucket(wc):
    if wc == 0 or pd.isna(wc): return "0. Empty/Missing"
    elif wc < 500: return "1. <500 (Thin)"
    elif wc < 1200: return "2. 500-1200 (Medium)"
    else: return "3. 1200+ (Detailed)"

df["wc_bucket"] = df["word_count"].apply(get_wc_bucket)
t2 = df.groupby("wc_bucket")["is_declining_label"].agg(["count", "mean"])
print(t2.to_string())

print("\n--- SIGNAL 3: Position Buckets vs. Decline ---")
def get_pos_bucket(pos):
    if pos == 0: return "0. Unranked"
    elif pos <= 10: return "1. Page 1 (1-10)"
    elif pos <= 20: return "2. Page 2 (10-20)"
    else: return "3. Page 3+ (20+)"

df["pos_bucket"] = df["avg_position"].apply(get_pos_bucket)
t3 = df.groupby("pos_bucket")["is_declining_label"].agg(["count", "mean"])
print(t3.to_string())


--- SIGNAL 1: Volume Buckets vs. Decline ---
                      count      mean
vol_bucket                           
1. <100 (Low)          7994  0.389042
2. 100-1000 (Medium)   8494  0.602896
3. 1000-10k (High)     9910  0.620081
4. 10k+ (Very High)    3602  0.523598

--- SIGNAL 2: Word Count Buckets vs. Decline ---
                      count      mean
wc_bucket                            
0. Empty/Missing       7699  0.465385
1. <500 (Thin)            3  0.333333
2. 500-1200 (Medium)   1291  0.249419
3. 1200+ (Detailed)   21007  0.588185

--- SIGNAL 3: Position Buckets vs. Decline ---
                   count      mean
pos_bucket                        
0. Unranked         1205  0.006639
1. Page 1 (1-10)   12983  0.563121
2. Page 2 (10-20)   7273  0.609515
3. Page 3+ (20+)    8539  0.528165


## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

### Flag Audited: Staleness behind Content Refresh Flags
- **Rule Assumption:** Pages that have not been updated in a long time (high `days_since_last_update`) are at higher risk of ranking decay and should be refreshed.
- **Verdict:** **MIXED**
- **Explanation:** The data shows that the decline rate does increase as staleness rises, peaking at 61.1% for the 90-180 day range. However, for pages older than 6 months (180d+), the decline rate drops back to 47.1%. Extremely old pages that still manage to rank represent stable evergreens, and the active decay is concentrated in the mid-term (3 to 6 months) window.

In [7]:
# Code cell 6: Run flag-linked staleness audit
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

print("--- FLAG SIGNAL AUDIT: Staleness vs. Decline ---")
def get_staleness_bucket(days):
    if days < 30: return "1. <30d"
    elif days < 90: return "2. 30-90d"
    elif days < 180: return "3. 90-180d"
    else: return "4. 180d+"

df["staleness_bucket"] = df["days_since_last_update"].apply(get_staleness_bucket)
t_stale = df.groupby("staleness_bucket")["is_declining_label"].agg(["count", "mean"])
print(t_stale.to_string())


--- FLAG SIGNAL AUDIT: Staleness vs. Decline ---
                  count      mean
staleness_bucket                 
1. <30d           20480  0.511377
2. 30-90d           175  0.588571
3. 90-180d         9171  0.611057
4. 180d+            174  0.471264


## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

### Key Takeaways for the Content Team:
1. **Target the Mid-Term Staleness Window:** Focus content updates on pages in the 3-6 month staleness range (90-180 days since last update) where decay is highest (61.1% decline rate), rather than blindly targeting the oldest pages (>180 days) which contain stable evergreen pieces.
2. **Prioritize Page 2 Content:** Focus refresh efforts on pages ranking in the 10-20 position range (Page 2), as they are the most volatile (61.0% decline rate) and are at high risk of slipping out of search visibility.
3. **Word Count is Not Quality:** Do not blindly write longer articles. In our dataset, longer articles (>1200 words) decay more often than medium articles (500-1200 words).

In [9]:
# Print signal audit verification summary
print("Signal audit complete. Findings suggest content updates should target mid-panel volatility rather than simple linear heuristics.")


Signal audit complete. Findings suggest content updates should target mid-panel volatility rather than simple linear heuristics.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.